# Práctica: Construcción de un modelo URDF

## Objetivo

Construir un modelo cinemático del manipulador **SO-100** a partir de las piezas STL proporcionadas, organizarlas y orientarlas en Blender, identificar las transformaciones entre sistemas de referencia y describir la estructura del robot mediante un archivo **URDF**. Finalmente, visualizar y verificar el modelo en VS Code mediante la extensión **URDF Visualizer**.

## Rúbrica de evaluación

### Metas

Que el alumno:

- Haga uso de funciones básicas de Blender
- Obtenga información a partir del modelo STL de un robot
- Visualice las transformaciones entre sistemas de referencia y las distinga de transformaciones de elementos visuales
- Construya un modelo URDF a partir de la información obtenida


### Contribución al perfil del egresado

La siguiente práctica contribuye en los siguientes puntos al perfil del egresado:


#### Aptitudes y habilidades

- Para diseñar, construir, operar y mantener los sistemas mecatrónicos y sus componentes.
- Para crear, innovar o evaluar las tecnologías relacionadas con la mecatrónica.


#### Actitudes profesionales

- Ser creativo e innovador.
- Tener confianza en su preparación académica.
- Comprometido con su actualización, superación y competencia profesional.


#### Actitudes de tipo social

- Promover el cambio en la mentalidad frente a la competitividad internacional.


### La evaluación de la práctica contará de los siguientes puntos

| Elemento | Porcentaje |
| ------:| ----------- |
| **Cuestionario previo** | 15% | 
| **Desarrollo** | 35% |
| **Análisis de resultados**  | 35% |
| **Conclusiones** | 15% |


### Se evaluará con los siguientes criterios:

| Elemento | Malo | Regular | Bueno |
| ------: | ------ | -------- | ------ |
| **Cuestionario previo** | El trabajo no contiene cuestionario previo o todas las preguntas son incorrectas (0%) | Al menos la mitad de las preguntas son correctas (8%) | Todas las preguntas son correctas (15%) |
| **Desarrollo** | El trabajo no contiene desarrollo o su planteamiento no concuerda con lo deseado (0%) | El desarrollo está mal planteado o no llega a los resultados esperados (10%) | El desarrollo tiene un planteamiento adecuado y llega a los resultados esperados (35%) |
| **Análisis de resultados** | El trabajo no contiene análisis de resultados o la información no se está interpretando correctamente (0%) | La interpretación de los resultados es parcial o desorganizada (10%) | Realiza un correcto análisis de los resultados de forma organizada (35%) |
| **Conclusiones** | El trabajo no contiene conclusiones o no hacen referencia al trabajo desarrollado y los objetivos planteados (0%) | La redacción de las conclusiones es desorganizada o confusa (8%) | Las conclusiones del trabajo son claras y hacen referencia al trabajo desarrollado y los objetivos planteados (15%) |


## Introducción

### Modelo geométrico y cinemático de un robot

Para describir un robot manipulador en ROS, se utiliza con frecuencia el formato **URDF (Unified Robot Description Format)**. Un archivo URDF es un documento XML que representa el robot como un conjunto de **eslabones** (*links*) conectados mediante **articulaciones** (*joints*). Cada eslabón puede incluir geometría visual, geometría de colisión y propiedades inerciales; cada junta define cómo se conecta un eslabón hijo con uno padre, el tipo de movimiento permitido y la pose relativa entre ambos sistemas de referencia [1].

Los tipos de junta más comunes son `fixed` (sin movimiento relativo), `revolute` (rotación limitada alrededor de un eje), `continuous` (rotación sin límites angulares) y `prismatic` (traslación). En un manipulador como el SO-100, las articulaciones de los motores se representan normalmente como articulaciones rotacionales. La etiqueta `origin` expresa la transformación del marco del eslabón padre al marco de la junta, que coincide con el marco del eslabón hijo cuando la articulación está en cero, mediante `xyz` (traslación) y `rpy` (orientación en ángulos roll, pitch y yaw, en radianes). El eje de movimiento se especifica mediante `axis`; por convención, se expresa en el marco de la junta [2].

Una descripción URDF no se obtiene únicamente colocando las mallas STL una junto a otra: es necesario determinar correctamente la relación entre los sistemas de referencia. Un error en la orientación de una junta puede hacer que el modelo se vea razonablemente bien, pero que el robot rote alrededor de un eje incorrecto. Por ello, en esta práctica se utilizará Blender para inspeccionar y preparar las piezas y, posteriormente, se describirán las transformaciones en URDF.

### Manejo básico de Blender para preparar piezas robóticas

**Blender** es una herramienta de modelado 3D que permite importar mallas STL, inspeccionar su geometría, transformar objetos y establecer jerarquías padre/hijo. Cada objeto tiene un sistema de referencia local (origen del objeto y ejes locales X, Y y Z), distinto del sistema de coordenadas global de la escena. Para esta práctica, los sistemas locales deben facilitar la interpretación cinemática: el eje **Z local de cada pieza asociada a un motor debe alinearse con el eje de giro de ese motor**.

Las transformaciones de un objeto incluyen ubicación, rotación y escala. En Blender, **Apply Rotation** aplica la rotación actual a los datos de la malla y deja los valores de rotación del objeto en cero, conservando su apariencia en la escena [3]. Esto resulta útil para preparar una geometría cuya orientación debe quedar incorporada en la malla, aunque debe hacerse deliberadamente: aplicar una rotación cambia cómo se interpretan las coordenadas locales de la geometría y puede afectar las transformaciones que luego se documenten en URDF.

En esta práctica, Blender se utilizará para construir un archivo de inspección y organización de las piezas, no para sustituir la definición cinemática del URDF. El modelo final deberá tener una jerarquía coherente y los marcos de referencia necesarios para calcular `origin xyz`, `origin rpy` y `axis`.

## Cuestionario previo

Responder brevemente antes de comenzar:

1. ¿Cuál es la diferencia entre un `link` y un `joint` en URDF?
> **Un `link` representa un eslabón o cuerpo rígido del robot y puede incluir su geometría visual, de colisión y propiedades inerciales. Un `joint` conecta dos eslabones y define el movimiento relativo permitido entre ellos.**

2. ¿Qué información se define en las etiquetas `origin`, `axis` y `parent`/`child` de una junta URDF?
> **`origin` define la posición (`xyz`) y orientación (`rpy`) del marco de la junta respecto al eslabón padre. `axis` indica la dirección del eje de movimiento expresada en el marco de la junta y `parent`/`child` identifican los eslabones conectados.**

3. ¿Cuál es la diferencia entre el sistema de coordenadas global y el sistema local de un objeto en Blender?
> **El sistema global es una referencia común y fija para toda la escena. El sistema local pertenece a cada objeto y depende de su posición y orientación.**

4. ¿Qué hace la operación **Apply Rotation** en blender?
> **Aplica la rotación del objeto a los datos de su malla y restablece a cero sus valores de rotación, conservando su apariencia visual.**

5. ¿En qué unidades se especifican las traslaciones y los ángulos en URDF?
> **Las traslaciones se expresan en metros (m) y los ángulos en radianes (rad).**

## Desarrollo

### 1. Preparación del espacio de trabajo

1. Dentro de la carpeta de la práctica, cree carpetas con la siguiente estructura. Guarde en `meshes/` los STL proporcionados y conserve los nombres de archivo originales cuando sea posible.

```text
practica_2/
├── blender/
│   └── so100_modelo.blend
├── meshes/
│   ├── Base.STL
│   ├── Shoulder_Rotation_Pitch.STL
│   ├── Upper_Arm.STL
│   ├── Lower_Arm.STL
│   ├── Wrist_Pitch_Roll.STL
│   ├── Fixed_Gripper.STL
│   └── Moving_Jaw.STL
├── urdf/
│   └── so100.urdf
└── imagenes/
    ├── p2_1.png
    ├── p2_2.png
    ├── blender_jerarquia.png
    ├── blender_ejes.png
    └── urdf_visualizer.png
```

2. Abra Blender y guarde el proyecto como `blender/so100_modelo.blend`.
3. Elimine los objetos iniciales que no vaya a utilizar.
4. Importe los archivos STL mediante **File → Import → STL** ó arrastrándolos hacia el editor de blender desde el explorador de archivos.


**Evidencia:** captura de la escena con todas las piezas importadas y nombres legibles.

**Datos disponibles para completar el desarrollo**

El URDF proporcionado define siete eslabones y seis articulaciones rotacionales. Los valores de las tablas siguientes se obtuvieron de ese código. Las operaciones realizadas en Blender, las unidades originales de las mallas y la validación visual quedan pendientes de comprobar con los archivos correspondientes.


### 2. Mostrar e inspeccionar los sistemas de referencia locales en Blender

1. Para visualizar los sistemas de referencia locales de cada pieza, abrir el menú deplegable de la parte superior derecha **"Viewport Gizmos"** y en la sección "**Object Gizmos**" habilitar *"Move"* y seleccionar en sistemas de referencia *"Local"*

<img src="imagenes/p2_1.png" align="center" alt="Viewport Gizmos" height="500"/>

2. En el menú **"Viewport Overlays"** habilite el eje *"z"* y ajuste la escala de la malla que representa al suelo a **0.001 [m]** (**1 mm**)

<img src="imagenes/p2_2.png" align="center" alt="Viewport Gizmos" height="500"/>

3. Seleccione cada pieza y observe dónde está su origen local y cómo están orientados sus ejes. Registre los resultados en una tabla.

**Información calculable a partir del URDF:** la segunda columna muestra la dirección del eje Z de cada marco respecto a la base, evaluada en la posición cero. La corrección que debe aplicarse en Blender requiere comparar esos marcos con los ejes y centros de giro de las mallas originales; por ello, la última columna queda pendiente.

| Pieza / eslabón | Eje Z del marco expresado en `base_link`, con todas las juntas en cero | Corrección necesaria del eje Z en Blender |
|---|---|---|
| Base / `base_link` | `0 0 1` | Pendiente de inspeccionar el STL original. |
| Hombro / `shoulder_link` | `0 0 1` | Pendiente de inspeccionar el STL original. |
| Brazo superior / `upper_arm_link` | `-1 0 0` | Pendiente de inspeccionar el STL original. |
| Brazo inferior / `lower_arm_link` | `-1 0 0` | Pendiente de inspeccionar el STL original. |
| Muñeca / `wrist_link` | `-1 0 0` | Pendiente de inspeccionar el STL original. |
| Pinza fija / `fixed_gripper_link` | `0 1 0` | Pendiente de inspeccionar el STL original. |
| Pinza móvil / `moving_jaw_link` | `0 0 -1` | Pendiente de inspeccionar el STL original. |


### 3. Orientar las piezas

La jerarquía en Blender sirve para organizar la escena; por sí sola no define el tipo de junta ni sustituye las transformaciones del URDF.

1. Rote la pieza hasta que el eje Z local asociado al motor coincida con el eje físico de giro. Puede utilizar `R` seguido del eje global o local y un ángulo, o introducir valores numéricos en el panel `N`. Confirme visualmente la alineación desde más de una vista.
2. Si se requiere que la orientación quede incorporada en la geometría, use **Object → Apply → Rotation** (`Ctrl+A` → **Rotation**) solo después de verificar el resultado. La rotación del objeto quedará en cero, pero la malla conservará la orientación visual. No aplique transformaciones de manera indiscriminada: registre qué piezas se modificaron y qué efecto tuvo en los ejes locales.
3. Repita el proceso para las piezas restantes. El eje Z del marco asociado a cada motor debe quedar alineado con el eje de giro correspondiente.
4. Guarde el archivo `.blend`.

**Importante:** no confunda “hacer que la malla se vea alineada” con “definir correctamente el marco de junta”. Para el URDF, además de la orientación de la malla visual, deben definirse de forma explícita el origen y el eje de cada junta. La geometría visual de un eslabón puede requerir una transformación propia respecto al marco del eslabón.

**Evidencias:** captura de una pieza con el eje *"z"* alineado con el eje del motor.

**Transformaciones visuales registradas en el URDF**

En los siete eslabones, `visual/origin xyz` es `0 0 0`. Las siguientes rotaciones ubican las mallas respecto a sus marcos de eslabón [1]:

| Pieza / eslabón | `visual/origin rpy` (rad) | Rotación visual definida |
|---|---|---|
| `base_link` | `0 0 0` | Sin rotación visual. |
| `shoulder_link` | `1.57079633 0 0` | +90° alrededor de X. |
| `upper_arm_link` | `0 1.57079633 0` | +90° alrededor de Y. |
| `lower_arm_link` | `0 1.57079633 0` | +90° alrededor de Y. |
| `wrist_link` | `0 1.57079633 0` | +90° alrededor de Y. |
| `fixed_gripper_link` | `1.57079633 0 0` | +90° alrededor de X. |
| `moving_jaw_link` | `0 0 0` | Sin rotación visual adicional. |

Estas transformaciones describen la geometría visual del URDF. No permiten confirmar que se ejecutó **Apply Rotation** en Blender. El registro de las piezas a las que se aplicó esa operación y sus efectos sobre los ejes locales queda pendiente [3].


### 4. Crear relaciones padre/hijo y obtener las transformaciones entre sistemas de referencia

La jerarquía en Blender sirve para organizar la escena; por sí sola no define el tipo de junta ni sustituye las transformaciones del URDF.

1. Determine la secuencia mecánica desde la base hasta el efector final. Para cada conexión, identifique cuál pieza será el padre y cuál el hijo.
2. Para establecer una relación padre/hijo, seleccione primero el objeto hijo, después el padre y presione `Ctrl+P`. Elija **Object (Keep Transform)** para conservar la posición y orientación actuales. Verifique la jerarquía en el *Outliner*. Si la opción o el resultado difieren según la versión, consulte el menú **Object → Parent**.

A partir del modelo preparado, determine para cada junta la transformación del marco padre al marco de la junta/hijo. Registre las traslaciones en metros y las orientaciones en radianes.

Una transformación homogénea se expresa como:
$$

{}^{P}\mathbf{T}_{J}=
\begin{bmatrix}
{}^{P}\mathbf{R}_{J} & {}^{P}\mathbf{p}_{J}\\
\mathbf{0}^{T} & 1
\end{bmatrix}

$$

donde \(P\) representa el marco padre, \(J\) el marco de la junta, \({}^{P}\mathbf{p}_{J}=[x\;y\;z]^T\) es la posición del origen de \(J\) respecto a \(P\), y \({}^{P}\mathbf{R}_{J}\) describe la orientación relativa.

Para cada conexión:

3. Identifique el centro del eje de giro del motor en la pieza padre.
4. Identifique la orientación que debe tener el marco de junta para que su eje Z coincida con el eje del motor.
5. Determine la posición relativa entre el origen del marco padre y el origen de la junta.
6. Exprese la orientación relativa mediante ángulos `roll`, `pitch`, `yaw` (en ese orden y en radianes para el atributo URDF `rpy`). No copie directamente los valores Euler de Blender sin verificar que representan la orientación relativa entre los marcos elegidos.
7. Si el origen del STL está desplazado del eje del motor, calcule la transformación necesaria para ubicar la geometría visual respecto al marco del eslabón. Esta transformación puede ser distinta de la transformación de la junta.

Complete la tabla siguiente. Use una fila por junta y añada filas para todas las articulaciones del SO-100 suministrado.

| Junta | Padre | Hijo | Tipo | `origin xyz` (m) | `origin rpy` (rad) | `axis` |
|---|---|---|---|---|---|---|
| `shoulder_joint` | `base_link` | `shoulder_link` | `revolute` | `0 -0.045 0.016` | `0 0 0` | `0 0 1` |
| `upper_arm_joint` | `shoulder_link` | `upper_arm_link` | `revolute` | `0 -0.0306 0.1025` | `0 -1.57079633 0` | `0 0 1` |
| `elbow_joint` | `upper_arm_link` | `lower_arm_link` | `revolute` | `0.028 0.11257 0` | `0 0 0` | `0 0 1` |
| `wrist_pitch_joint` | `lower_arm_link` | `wrist_link` | `revolute` | `0.1349 0.0052 0` | `0 0 0` | `0 0 1` |
| `wrist_roll_joint` | `wrist_link` | `fixed_gripper_link` | `revolute` | `0.00055 -0.0601 0.001088` | `-1.57079633 1.57079633 0` | `0 0 1` |
| `gripper_joint` | `fixed_gripper_link` | `moving_jaw_link` | `revolute` | `-0.020292 -0.0001 -0.024312` | `1.57079633 0 3.14159265` | `0 0 1` |

Los datos se extrajeron de las etiquetas `<joint>` del URDF proporcionado. Cada `axis` está expresado en el marco de su junta; el vector `0 0 1` representa su eje Z local.

**Ejemplo de obtención de una traslación: articulación del codo**

Según las posiciones anotadas en el código, el desplazamiento del brazo superior al inferior, expresado en la base, es:

$$
\Delta\mathbf{p}^{B} =
\begin{bmatrix}0\\0.03697\\0.1465\end{bmatrix}
-\begin{bmatrix}0\\-0.0756\\0.1185\end{bmatrix}
=\begin{bmatrix}0\\0.11257\\0.028\end{bmatrix}\ \mathrm{m}.
$$

Como el marco padre tiene una rotación de aproximadamente $-\pi/2$ alrededor de Y respecto a la base, se expresa ese desplazamiento en el marco padre mediante la rotación inversa:

$$
{}^{P}\mathbf{p}_{J} = ({}^{B}\mathbf{R}_{P})^{T}\Delta\mathbf{p}^{B}
=\begin{bmatrix}0.028\\0.11257\\0\end{bmatrix}\ \mathrm{m}.
$$

Se obtiene así el valor `origin xyz="0.028 0.11257 0"` de `elbow_joint`.

**Evidencias:** captura del *Outliner* con las jerarquías padre/hijo y el modelo del robot construído en blender

### 5. Construir el archivo URDF en Visual Studio Code

1. Abra en VS Code la carpeta `practica_2/`.
2. Cree `urdf/so100.urdf`.
3. Declare el robot con una etiqueta raíz `<robot name="so100">`.
4. Defina un `<link>` por cada eslabón físico. Incluya una etiqueta `<visual>` con `<geometry><mesh filename="..."/></geometry>`. Use rutas relativas al archivo URDF, por ejemplo `filename="../meshes/Base.STL"`; compruebe que el visualizador resuelva correctamente las rutas. Si la extensión requiere una organización distinta, ajuste las rutas y documente el cambio. Para el nombre de las etiquetas de eslabón, puede usar los nombres de los archivos stl
5. Para cada unión entre eslabones, cree una etiqueta `<joint>` con `name`, `type`, `parent`, `child`, `origin` y `axis`. Añada `limit` para articulaciones `revolute`.
6. Use `base_link` como eslabón raíz y conecte todas las piezas hasta la pinza móvil. Cada eslabón hijo debe conectarse mediante una junta, y cada eslabón debe tener una única conexión padre dentro del árbol cinemático.
7. Incluya una geometría visual por pieza STL. Si el origen local de la malla no coincide con el marco del eslabón, utilice los atributos `origin xyz` y `origin rpy` dentro de `<visual>` para posicionar la malla, sin confundirlos con los de `<joint>`.
8. Para los archivos STL, revise si las unidades de origen son milímetros. URDF interpreta las coordenadas de `origin xyz` en metros; además, verifique que la escala de las mallas importadas sea coherente. No compense un error de escala con traslaciones.
9. Mantenga el XML bien formado: todas las etiquetas deben cerrarse y los nombres de los `link` usados en `parent` y `child` deben existir.

**Código URDF completo**

Se incorporó el código proporcionado, conservando sus transformaciones, límites, colores y escalas. Las rutas se ajustaron a la estructura de carpetas indicada: el URDF se guarda en `urdf/` y los STL en `meshes/`. El nombre de la pinza móvil se escribió como `Moving_Jaw.STL`, sin la barra invertida que aparecía antes del punto en el texto pegado; debe coincidir con el nombre real de la malla.

```xml
<?xml version="1.0" encoding="utf-8"?>

<robot name="so100">

    <!-- 1. Base -->
    <!-- Posición de referencia anotada en Blender (m), con q=0: 0 0 0 -->
    <!-- Corrección visual del STL: Sin rotación visual -->
    <link name="base_link">
        <visual>
            <origin xyz="0 0 0" rpy="0 0 0"/>
            <geometry>
                <mesh filename="../meshes/Base.STL" scale="1 1 1"/>
            </geometry>
            <material name="negro">
                <color rgba="0.1 0.1 0.1 1.0"/>
            </material>
        </visual>
    </link>

    <!-- 2. Hombro -->
    <!-- Posición de referencia anotada en Blender (m), con q=0: 0 -0.045 0.016 -->
    <!-- Corrección visual del STL: +90° alrededor de X -->
    <link name="shoulder_link">
        <visual>
            <origin xyz="0 0 0" rpy="1.57079633 0 0"/>
            <geometry>
                <mesh filename="../meshes/Shoulder_Rotation_Pitch.STL" scale="1 1 1"/>
            </geometry>
            <material name="rojo">
                <color rgba="1.0 0.0 0.0 1.0"/>
            </material>
        </visual>
    </link>

    <!-- Articulación 1: Base y hombro: giro del hombro -->
    <joint name="shoulder_joint" type="revolute">
        <origin xyz="0 -0.045 0.016" rpy="0 0 0"/>
        <parent link="base_link"/>
        <child link="shoulder_link"/>
        <axis xyz="0 0 1"/>
        <limit lower="-1.96" upper="1.96" effort="1000" velocity="5.0"/>
    </joint>

    <!-- 3. Brazo superior -->
    <!-- Posición de referencia anotada en Blender (m), con q=0: 0 -0.0756 0.1185 -->
    <!-- Corrección visual del STL: +90° alrededor de Y -->
    <link name="upper_arm_link">
        <visual>
            <origin xyz="0 0 0" rpy="0 1.57079633 0"/>
            <geometry>
                <mesh filename="../meshes/Upper_Arm.STL" scale="1 1 1"/>
            </geometry>
            <material name="azul">
                <color rgba="0.0 0.3 1.0 1.0"/>
            </material>
        </visual>
    </link>

    <!-- Articulación 2: Hombro y brazo superior: flexión del brazo -->
    <joint name="upper_arm_joint" type="revolute">
        <origin xyz="0 -0.0306 0.1025" rpy="0 -1.57079633 0"/>
        <parent link="shoulder_link"/>
        <child link="upper_arm_link"/>
        <axis xyz="0 0 1"/>
        <limit lower="-1.745" upper="1.745" effort="1000" velocity="5.0"/>
    </joint>

    <!-- 4. Brazo inferior -->
    <!-- Posición de referencia anotada en Blender (m), con q=0: 0 0.03697 0.1465 -->
    <!-- Corrección visual del STL: +90° alrededor de Y -->
    <link name="lower_arm_link">
        <visual>
            <origin xyz="0 0 0" rpy="0 1.57079633 0"/>
            <geometry>
                <mesh filename="../meshes/Lower_Arm.STL" scale="1 1 1"/>
            </geometry>
            <material name="verde">
                <color rgba="0.0 0.8 0.2 1.0"/>
            </material>
        </visual>
    </link>

    <!-- Articulación 3: Brazo superior e inferior: flexión del codo -->
    <joint name="elbow_joint" type="revolute">
        <origin xyz="0.028 0.11257 0" rpy="0 0 0"/>
        <parent link="upper_arm_link"/>
        <child link="lower_arm_link"/>
        <axis xyz="0 0 1"/>
        <limit lower="-1.5" upper="1.5" effort="1000" velocity="5.0"/>
    </joint>

    <!-- 5. Muñeca -->
    <!-- Posición de referencia anotada en Blender (m), con q=0: 0 0.04217 0.2814 -->
    <!-- Corrección visual del STL: +90° alrededor de Y -->
    <link name="wrist_link">
        <visual>
            <origin xyz="0 0 0" rpy="0 1.57079633 0"/>
            <geometry>
                <mesh filename="../meshes/Wrist_Pitch_Roll.STL" scale="1 1 1"/>
            </geometry>
            <material name="amarillo">
                <color rgba="1.0 0.85 0.0 1.0"/>
            </material>
        </visual>
    </link>

    <!-- Articulación 4: Brazo inferior y muñeca: inclinación de la muñeca -->
    <joint name="wrist_pitch_joint" type="revolute">
        <origin xyz="0.1349 0.0052 0" rpy="0 0 0"/>
        <parent link="lower_arm_link"/>
        <child link="wrist_link"/>
        <axis xyz="0 0 1"/>
        <limit lower="-1.658" upper="1.658" effort="1000" velocity="5.0"/>
    </joint>

    <!-- 6. Pinza fija -->
    <!-- Posición de referencia anotada en Blender (m), con q=0: -0.001088 -0.01793 0.28195 -->
    <!-- Corrección visual del STL: +90° alrededor de X -->
    <link name="fixed_gripper_link">
        <visual>
            <origin xyz="0 0 0" rpy="1.57079633 0 0"/>
            <geometry>
                <mesh filename="../meshes/Fixed_Gripper.STL" scale="1 1 1"/>
            </geometry>
            <material name="morado">
                <color rgba="0.6 0.1 0.9 1.0"/>
            </material>
        </visual>
    </link>

    <!-- Articulación 5: Muñeca y pinza fija: giro longitudinal de la pinza -->
    <joint name="wrist_roll_joint" type="revolute">
        <origin xyz="0.00055 -0.0601 0.001088" rpy="-1.57079633 1.57079633 0"/>
        <parent link="wrist_link"/>
        <child link="fixed_gripper_link"/>
        <axis xyz="0 0 1"/>
        <limit lower="-2.75" upper="2.75" effort="1000" velocity="5.0"/>
    </joint>

    <!-- 7. Pinza móvil -->
    <!-- Posición de referencia anotada en Blender (m), con q=0: -0.02138 -0.042242 0.28205 -->
    <!-- Corrección visual del STL: Sin rotación visual adicional -->
    <link name="moving_jaw_link">
        <visual>
            <origin xyz="0 0 0" rpy="0 0 0"/>
            <geometry>
                <mesh filename="../meshes/Moving_Jaw.STL" scale="1 1 1"/>
            </geometry>
            <material name="naranja">
                <color rgba="1.0 0.4 0.0 1.0"/>
            </material>
        </visual>
    </link>

    <!-- Articulación 6: Pinza fija y móvil: apertura y cierre -->
    <joint name="gripper_joint" type="revolute">
        <origin xyz="-0.020292 -0.0001 -0.024312" rpy="1.57079633 0 3.14159265"/>
        <parent link="fixed_gripper_link"/>
        <child link="moving_jaw_link"/>
        <axis xyz="0 0 1"/>
        <limit lower="-0.1792" upper="1.5708" effort="1000" velocity="5.0"/>
    </joint>

</robot>
```

La escala `1 1 1` y los valores de `limit` se conservaron del código recibido; no se han validado como dimensiones o especificaciones físicas del robot. Su comprobación con las mallas y la documentación del equipo queda pendiente.


### 6. Visualizar y validar el URDF con URDF Visualizer

1. Compruebe en el visualizador de URDF en VSCode: 
   - Todas las piezas aparecen y tienen una escala razonable.
   - La base está en una posición coherente.
   - Las piezas están conectadas en el orden cinemático correcto.
   - Los ejes de las articulaciones coinciden con los ejes de giro de los motores.
   - Al modificar la posición de una junta desde el visualizador, las piezas hijas se mueven alrededor del eje esperado.
   - No hay piezas inesperadamente desplazadas, giradas o reflejadas.
2. Si una pieza aparece orientada incorrectamente, determine si el error está en la transformación de la junta (`joint/origin`), en la transformación de la malla (`visual/origin`) o en la orientación/escala de la propia malla. Corrija el parámetro correspondiente en vez de introducir compensaciones arbitrarias.
3. Guarde el archivo URDF y las evidencias. Tome una captura del robot completo.

**Entregables del desarrollo**

- Archivo Blender `so100_modelo.blend` con las piezas organizadas, nombres descriptivos y jerarquía padre/hijo.
- Archivo `so100.urdf` con los links, joints, mallas y transformaciones.
- Tabla de transformaciones entre marcos de referencia.
- Capturas de los ejes locales, la jerarquía de Blender y el robot visualizado en VS Code.

**Verificación estática del código**

| Comprobación | Estado |
|---|---|
| XML bien formado y etiqueta raíz `robot` | Verificado mediante lectura del XML. |
| Siete eslabones y seis articulaciones `revolute` | Verificado. |
| Existencia de los eslabones indicados en `parent` y `child` | Verificado. |
| Una sola raíz, `base_link`, y una conexión padre para cada eslabón restante | Verificado. |
| Cadena conectada sin ciclos y límites definidos para todas las juntas | Verificado. |
| Posiciones de los marcos en la posición cero | Coinciden con las posiciones de Blender anotadas en los comentarios del código, dentro del redondeo numérico. |
| Carga y escala real de los STL | Pendiente de comprobar en el visualizador. |
| Coincidencia de los ejes con los motores y movimiento de cada articulación | Pendiente de comprobar en el visualizador. |

Esta revisión estática no sustituye la inspección del archivo Blender ni la prueba de movimiento del modelo.


## Análisis de resultados

Responder con base en las decisiones y verificaciones realizadas:

1. ¿Qué diferencias hay entre los sistemas de referencia globales y locales de Blender? ¿Cómo los utilizó para identificar los ejes de los motores?
> **RESPUESTA**

2. ¿Qué efecto tuvo aplicar una rotación a la malla y cómo comprobó que no se alteró la interpretación cinemática de las piezas?
> **RESPUESTA**

3. Explique la diferencia entre la transformación definida en `joint/origin` y la definida en `visual/origin`.
> **RESPUESTA**

4. ¿Qué ventajas tiene describir el robot mediante URDF frente a disponer únicamente de un conjunto de modelos STL?
> **RESPUESTA**

## Conclusiones

> **CONCLUSIONES**

## Bibliografía

Incluya las fuentes consultadas en formato IEEE, en caso de haberlas consultado.

[1] ROS 2, “Building a visual robot model from scratch — tutorial,” *ROS 2 Documentation*, repositorio `ros2_documentation`, rama `rolling`. [En línea]. Disponible: [https://github.com/ros2/ros2_documentation/blob/rolling/source/Capabilities/Simulation/URDF/Building-a-Visual-Robot-Model-with-URDF-from-Scratch.rst](https://github.com/ros2/ros2_documentation/blob/rolling/source/Capabilities/Simulation/URDF/Building-a-Visual-Robot-Model-with-URDF-from-Scratch.rst). [Consultado: 6-nov-2026].

[2] ROS 2, “Building a movable robot model — tutorial,” *ROS 2 Documentation*, repositorio `ros2_documentation`, rama `rolling`. [En línea]. Disponible: [https://github.com/ros2/ros2_documentation/blob/rolling/source/Capabilities/Simulation/URDF/Building-a-Movable-Robot-Model-with-URDF.rst](https://github.com/ros2/ros2_documentation/blob/rolling/source/Capabilities/Simulation/URDF/Building-a-Movable-Robot-Model-with-URDF.rst). [Consultado: 6-nov-2026].

[3] Blender Foundation, “Apply,” *Blender 4.5 LTS Manual*. [En línea]. Disponible: [https://docs.blender.org/manual/en/4.5/scene_layout/object/editing/apply.html](https://docs.blender.org/manual/en/4.5/scene_layout/object/editing/apply.html). [Consultado: 6-nov-2026].
